# Bayesian Inference

Companion notebook for the [Bayesian Inference](https://ml-viz.vercel.app/courses/probability-statistics/04-bayesian-inference) lesson.

We'll visualize posterior updating with conjugate priors and show the MAP-regularization connection.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#30344a', 'text.color': '#e2e8f0',
    'axes.labelcolor': '#e2e8f0', 'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8',
})

## Bayes' theorem on a concrete event — the medical test

A disease affects 1% of the population. The test is 99% sensitive (`P(+|D)=0.99`)
and 95% specific, so `P(+|not D)=0.05`. You test positive. What is `P(D|+)`?

We compute the evidence `P(+)` by summing the true-positive and false-positive
paths, then divide — exactly the steps from the lesson.

In [ ]:
# Given probabilities
p_D        = 0.01    # prior: prevalence of disease
p_pos_D    = 0.99    # sensitivity:  P(+ | disease)
p_pos_notD = 0.05    # false-positive rate: P(+ | healthy) = 1 - specificity
p_notD     = 1 - p_D

# Step 1: evidence  P(+) = true positives + false positives
true_pos  = p_pos_D    * p_D
false_pos = p_pos_notD * p_notD
p_pos     = true_pos + false_pos

# Step 2: posterior via Bayes' theorem
p_D_given_pos = true_pos / p_pos

print("True-positive  mass  P(+,D)     = {:.4f}".format(true_pos))
print("False-positive mass  P(+,notD)  = {:.4f}".format(false_pos))
print("Evidence             P(+)       = {:.4f}".format(p_pos))
print("Posterior            P(D|+)     = {:.4f}  (~{:.0f}%)".format(
    p_D_given_pos, 100 * p_D_given_pos))
print()
print("Despite a '99% accurate' test, a positive result means only ~17% chance of disease.")
print("The tiny prior (1%) lets false positives from the large healthy group dominate.")

# Sanity-check: how the posterior climbs as the disease becomes more common
for prev in [0.01, 0.05, 0.10, 0.50]:
    tp = p_pos_D * prev
    fp = p_pos_notD * (1 - prev)
    print("prevalence={:>5.0%}  ->  P(D|+) = {:.3f}".format(prev, tp / (tp + fp)))

## Conjugate updating — Beta-Bernoulli

Prior `p ~ Beta(a, b)`, flips `x_i ~ Bernoulli(p)`. After `k` heads in `n` flips the
posterior is `Beta(a + k, b + n - k)` (derived step-by-step in the lesson).

First we verify the closed forms on one batch (`Beta(2,2)` prior, 7 heads of 10),
then we plot the belief updating sequentially.

In [ ]:
# --- Closed-form check on the lesson's numeric instance ---
a0, b0 = 2, 2          # Beta(2, 2) prior
k, n   = 7, 10         # 7 heads, 3 tails
a_post, b_post = a0 + k, b0 + (n - k)   # -> Beta(9, 5)

post_mean = a_post / (a_post + b_post)               # (a+k)/(a+b+n)
post_mode = (a_post - 1) / (a_post + b_post - 2)      # mode of Beta(a,b)
p_mle     = k / n

print("Posterior = Beta({}, {})".format(a_post, b_post))
print("Posterior mean (point estimate)      = {:.3f}".format(post_mean))   # 0.643
print("Posterior mode (MAP estimate)        = {:.3f}".format(post_mode))   # 0.667
print("MLE  k/n (data only, prior ignored)  = {:.3f}".format(p_mle))       # 0.700
print("Prior mode = 0.500  ->  MAP sits between prior and MLE")

# --- Plot prior vs posterior ---
p_range = np.linspace(0.001, 0.999, 500)
prior_pdf = stats.beta.pdf(p_range, a0, b0)
post_pdf  = stats.beta.pdf(p_range, a_post, b_post)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(p_range, prior_pdf, color='#94a3b8', lw=2, label='Prior  Beta(2, 2)')
ax.fill_between(p_range, prior_pdf, alpha=0.15, color='#94a3b8')
ax.plot(p_range, post_pdf, color='#6366f1', lw=2.5, label='Posterior  Beta(9, 5)')
ax.fill_between(p_range, post_pdf, alpha=0.20, color='#6366f1')
ax.axvline(p_mle,     color='#f97316', lw=1.5, ls='--', label='MLE = {:.2f}'.format(p_mle))
ax.axvline(post_mean, color='#2dd4bf', lw=1.5, ls=':',  label='Post. mean = {:.2f}'.format(post_mean))
ax.set_xlabel('p (coin bias)'); ax.set_ylabel('density')
ax.set_title('Prior updated to posterior after 7 heads / 10 flips')
ax.legend(); ax.grid(True, alpha=0.2)
plt.tight_layout(); plt.show()

## Sequential Bayesian updating

Each new flip turns the previous posterior into the new prior. We watch the belief
sharpen as observations accumulate — the posterior standard deviation (`sd`) shrinks
and the density concentrates on the true bias.

In [ ]:
rng = np.random.default_rng(42)
true_p = 0.7
n_obs  = 50
data   = rng.binomial(1, true_p, n_obs)

# Prior: Beta(2, 2) — mild belief coin is fair
alpha0, beta0 = 2, 2

p_range = np.linspace(0.001, 0.999, 500)
snapshots = [0, 1, 5, 20, 50]

fig, axes = plt.subplots(1, len(snapshots), figsize=(18, 4), sharey=False)
colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(snapshots)))

for i, n in enumerate(snapshots):
    n_heads = int(data[:n].sum()) if n > 0 else 0
    n_tails = n - n_heads
    alpha_post = alpha0 + n_heads
    beta_post  = beta0  + n_tails

    pdf = stats.beta.pdf(p_range, alpha_post, beta_post)
    post_mean = alpha_post / (alpha_post + beta_post)
    post_std  = stats.beta.std(alpha_post, beta_post)   # spread = uncertainty

    axes[i].plot(p_range, pdf, color=colors[i], lw=2)
    axes[i].fill_between(p_range, pdf, alpha=0.2, color=colors[i])
    axes[i].axvline(true_p, color='#2dd4bf', lw=1.5, linestyle=':', label='True p')
    axes[i].axvline(post_mean, color='#f97316', lw=1.5, linestyle='--',
                    label='mean={:.2f}'.format(post_mean))
    axes[i].set_title('After {} obs\n(H={}, T={}), sd={:.3f}'.format(
        n, n_heads, n_tails, post_std), fontsize=9)
    axes[i].set_xlabel('p'); axes[i].grid(True, alpha=0.2)
    axes[i].legend(fontsize=7)

axes[0].set_ylabel('Posterior density')
plt.suptitle('Bayesian updating of coin bias belief (true p={})'.format(true_p),
             y=1.02, fontsize=12)
plt.tight_layout(); plt.show()

print('As n grows the posterior narrows (sd shrinks) and concentrates on true p={}.'.format(true_p))

## MAP = L2 regularization (Gaussian prior on weights)

In [ ]:
rng = np.random.default_rng(1)
n = 20
X = np.column_stack([np.ones(n), rng.uniform(-2, 2, n)])
true_w = np.array([1.0, 3.0])
y = X @ true_w + rng.normal(0, 0.5, n)

def mle_solution(X, y):
    return np.linalg.solve(X.T @ X, X.T @ y)

def map_solution(X, y, lam):
    # MAP with Gaussian prior N(0, tau^2): lam = 1/(2 tau^2)
    return np.linalg.solve(X.T @ X + lam * np.eye(2), X.T @ y)

w_mle = mle_solution(X, y)
lambdas = [0.1, 1.0, 5.0, 20.0]

print('True weights: {}'.format(true_w))
print('MLE:          {}'.format(w_mle.round(4)))
print()
for lam in lambdas:
    w_map = map_solution(X, y, lam)
    print('MAP (lam={:5.1f} = 1/2tau^2): {}'.format(lam, w_map.round(4)))

print('\nNote: as lam -> inf, MAP -> [0, 0] (prior dominates)')
print('      as lam -> 0,   MAP -> MLE   (data dominates)')

# Plot shrinkage of w0 and w1 vs lambda
lam_range = np.logspace(-2, 2, 100)
w0_path = [map_solution(X, y, l)[0] for l in lam_range]
w1_path = [map_solution(X, y, l)[1] for l in lam_range]

fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogx(lam_range, w0_path, color='#6366f1', lw=2, label='w0 (intercept)')
ax.semilogx(lam_range, w1_path, color='#f97316', lw=2, label='w1 (slope)')
ax.axhline(true_w[0], color='#6366f1', lw=1, linestyle=':', alpha=0.7, label='True w0')
ax.axhline(true_w[1], color='#f97316', lw=1, linestyle=':', alpha=0.7, label='True w1')
ax.set_xlabel('lambda (regularization strength)'); ax.set_ylabel('Weight value')
ax.set_title('MAP weight shrinkage as lambda increases (L2 / Ridge)')
ax.legend(); ax.grid(True, alpha=0.2)
plt.tight_layout(); plt.show()

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — Bayes' theorem for a diagnostic test

Put the medical-test example into one reusable function:

$$P(D \mid +) = \frac{P(+ \mid D)\,P(D)}{P(+ \mid D)\,P(D) + P(+ \mid \neg D)\,P(\neg D)}$$

The checks include the classic surprise (1% base rate + a 99%-sensitive test → only ~17% posterior) and two sanity limits: a test with **zero false positives** makes a positive certain, and a test whose positive rate is identical for sick and healthy is **uninformative** — the posterior equals the prior.

In [ ]:
def posterior(prior, sensitivity, false_positive_rate):
    """P(disease | positive test)."""
    # TODO(you): total probability of testing positive
    # (sick and caught) + (healthy but false-positive)
    p_pos = ...

    # TODO(you): Bayes' theorem: (sensitivity * prior) / p_pos
    return ...

In [ ]:
# Checks — run me
assert abs(posterior(0.01, 0.99, 0.05) - 0.0099 / (0.0099 + 0.0495)) < 1e-12, \
    "1% base rate, 99% sensitivity, 5% false positives -> ~16.7%"
assert posterior(0.01, 0.99, 0.05) < 0.2, "base rates dominate rare events"
assert abs(posterior(0.3, 0.9, 0.0) - 1.0) < 1e-12, "no false positives -> a positive is certain"
assert abs(posterior(0.3, 0.7, 0.7) - 0.3) < 1e-12, "uninformative test -> posterior = prior"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def posterior(prior, sensitivity, false_positive_rate):
    p_pos = sensitivity * prior + false_positive_rate * (1 - prior)
    return sensitivity * prior / p_pos
```

</details>

### Exercise 2 — Beta-Bernoulli updating

Conjugacy makes the posterior update pure bookkeeping: starting from $\text{Beta}(\alpha, \beta)$ and observing coin flips,

$$\alpha \leftarrow \alpha + \#\text{heads}, \qquad \beta \leftarrow \beta + \#\text{tails}, \qquad \mathbb{E}[p] = \frac{\alpha}{\alpha + \beta}$$

The last check verifies the property that makes streaming updates work: updating **one flip at a time** gives exactly the same posterior as updating with the whole batch.

In [ ]:
def beta_update(alpha, beta, flips):
    """Posterior (alpha, beta) after observing flips (1 = heads, 0 = tails)."""
    flips = np.asarray(flips)

    # TODO(you): count the 1s and the 0s
    heads = ...
    tails = ...

    return alpha + heads, beta + tails


def beta_mean(alpha, beta):
    # TODO(you): the posterior mean alpha / (alpha + beta)
    return ...

In [ ]:
# Checks — run me
a, b = beta_update(1, 1, [1, 1, 1, 0, 1, 1, 0, 1, 0, 1])
assert (a, b) == (8, 4), "uniform prior + 7 heads / 3 tails -> Beta(8, 4)"
assert abs(beta_mean(a, b) - 2 / 3) < 1e-12, "posterior mean 8/12 = 2/3"

a1, b1 = beta_update(*beta_update(2, 5, [1, 0]), [1, 1])
a2, b2 = beta_update(2, 5, [1, 0, 1, 1])
assert (a1, b1) == (a2, b2), "sequential and batch updates must agree"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def beta_update(alpha, beta, flips):
    flips = np.asarray(flips)
    heads = int(np.sum(flips == 1))
    tails = int(np.sum(flips == 0))
    return alpha + heads, beta + tails


def beta_mean(alpha, beta):
    return alpha / (alpha + beta)
```

</details>